# 04 · Dimensiones restantes y carga de `fact_ventas`
**IDY1102 · EP3 · DataRetail 360**

| Campo | Detalle |
|---|---|
| Responsable | [Nombre del integrante líder de integración] (el trabajo se reparte entre los tres) |
| Se ejecuta | **Al final**, cuando 01, 02 y 03 ya cargaron todos los snapshots |
| Dimensiones simples | `dim_fecha` (ejemplo resuelto), `dim_canal`, `dim_sucursal` (SCD Tipo 1) |
| Hecho | `fact_ventas`, grano: una línea de producto por boleta u orden |
| Fuentes | `ventas_pos.csv` (tiendas) y `ventas_ecommerce.json` (canal digital, anidado) |

> Aquí se une todo: cada línea de venta busca la **versión de la dimensión vigente a la fecha de la venta**
> (`db.asignar_clave_scd2`). Las celdas **✏️ TODO** son las reglas de calidad y las búsquedas de claves.

## 0. Setup

In [ ]:
import os, sys
from datetime import date
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

from src import db, extract, medallion, quality as q, schema
from src.config import get_settings
from src.logger import get_logger

PROCESO = "integracion"
log = get_logger(PROCESO)
cfg = get_settings()
engine = db.get_engine(cfg)
schema.crear_esquema(engine)                  # idempotente: crea solo lo que falta
schema.sembrar_miembros_desconocidos(engine)  # miembro -1 "No informado" en cada dimensión
pd.set_option("display.width", 160)

## A. `dim_fecha` — ejemplo resuelto
Se genera (no viene de una fuente). Sirve de modelo para el patrón *extract → transform → load → validar*.

In [ ]:
def construir_dim_fecha(desde: str = "2024-01-01", hasta: str = "2026-12-31") -> pd.DataFrame:
    f = pd.DataFrame({"fecha": pd.date_range(desde, hasta)})
    f["fecha_key"] = f["fecha"].dt.strftime("%Y%m%d").astype(int)      # clave inteligente AAAAMMDD
    f["dia"] = f["fecha"].dt.day
    f["mes"] = f["fecha"].dt.month
    f["nombre_mes"] = f["fecha"].dt.month_name()
    f["trimestre"] = f["fecha"].dt.quarter
    f["anio"] = f["fecha"].dt.year
    f["dia_semana"] = f["fecha"].dt.day_name()
    f["es_fin_de_semana"] = (f["fecha"].dt.dayofweek >= 5).astype(int)
    return f

with db.Ejecucion(engine, "dim_fecha") as run:
    fechas = construir_dim_fecha()
    run.leidas = len(fechas)
    r = db.insertar_si_no_existe(engine, schema.dim_fecha, fechas, ["fecha_key"])   # idempotente
    run.cargadas = r["insertadas"]
r

## B. `dim_canal`  ✏️ TODO
Dimensión Tipo 1 con pocos valores. Los canales de las fuentes vienen escritos de varias formas (`Tienda física`, `TIENDA FÍSICA`, `tienda fisica`, `E-commerce`, `ecommerce`, `Web`).

In [ ]:
def cargar_dim_canal() -> dict:
    """✏️ TODO: definir los canales canónicos y cargarlos con `db.aplicar_scd1(engine, "dim_canal", df, "canal", ["tipo_canal"], "canal_key")`.

    Define también la función que lleva cualquier variante de la fuente a su canal canónico: la usarás en `transformar_ventas`.
    """
    raise NotImplementedError("TODO: cargar dim_canal")

## C. `dim_sucursal`  ✏️ TODO
Lee `sucursales.csv`, resuelve `geografia_key` desde `codigo_comuna` (`dim_geografia` ya cargada) y carga con SCD Tipo 1.

In [ ]:
def cargar_dim_sucursal() -> dict:
    """✏️ TODO: extraer `sucursales.csv`, mapear `geografia_key` (`db.mapa_claves`) y cargar con `db.aplicar_scd1`.

    Atributos: nombre_sucursal, tipo_tienda, geografia_key. Clave natural: id_sucursal.
    """
    raise NotImplementedError("TODO: cargar dim_sucursal")

## D. `fact_ventas`

### D.1 Extract
Une las dos fuentes en un solo esquema. El JSON de e-commerce viene anidado (`items` dentro de cada orden) y se aplana con `leer_json`.

In [ ]:
COLUMNAS_COMUNES = ["id_transaccion", "linea", "fecha", "sku", "rut", "id_sucursal", "canal",
                    "cantidad", "monto_total", "estado", "origen_archivo", "fila_origen"]

def extraer_ventas() -> pd.DataFrame:
    """Lee POS (CSV) y e-commerce (JSON) y las deja en el mismo esquema. Guarda el crudo en Bronze."""
    pos = extract.leer_csv(cfg.data_dir / "ventas_pos.csv").rename(columns={"monto": "monto_total"})
    ec = extract.leer_json(cfg.data_dir / "ventas_ecommerce.json", record_path="items",
                           meta=["order_id", "created_at", "channel", "status", ["customer", "rut"]])
    ec = ec.rename(columns={"order_id": "id_transaccion", "created_at": "fecha", "channel": "canal",
                            "qty": "cantidad", "line_total": "monto_total", "customer_rut": "rut"})
    ec["id_sucursal"] = "S00"                                   # el canal digital sale del centro de distribución
    ec["estado"] = ec["status"].map({"PAID": "OK", "RETURNED": "DEVUELTA", "CANCELLED": "ANULADA"})
    # Las líneas se numeran DENTRO de cada aparición de la orden: si una orden viene repetida, sus líneas
    # repiten (id_transaccion, linea) y la regla de líneas duplicadas del Transform las detecta.
    ec["linea"] = (ec.groupby(["id_transaccion", "_registro"]).cumcount() + 1).astype("string")   # Bronze = todo texto
    crudo = pd.concat([pos[COLUMNAS_COMUNES], ec[COLUMNAS_COMUNES]], ignore_index=True)
    medallion.guardar_capa(crudo, "bronze", "ventas")
    return crudo

### D.2 Transform  ✏️ TODO
Reglas de calidad que traen los datos de ejemplo:

| Problema en los datos de ejemplo | Dimensión de calidad | Pista |
|---|---|---|
| Líneas repetidas en el POS (misma `id_transaccion` + `linea`, 2 filas) | Unicidad | `duplicated(["id_transaccion", "linea"])` |
| Una orden de e-commerce aparece dos veces en el JSON (`EC-1002`) | Unicidad | sus líneas repiten `(id_transaccion, linea)`: la misma regla las atrapa (ver cómo numera `extraer_ventas`) |
| Fechas en 4 formatos (y una imposible, `31-02-2025`) | Validez | `q.parsear_fechas` y rechazar las `NaT` |
| Montos negativos con estado OK (2 filas) | Validez | `monto_total < 0` → rechazo |
| Ventas anuladas (POS y e-commerce) | Consistencia con el negocio | decidir y documentar: ¿se rechazan o se cargan marcadas? |
| Devoluciones | Consistencia | columna `devolucion` 0/1 (como en el Diccionario 2.1.5) |
| SKU escritos distinto, y un SKU que no existe en el catálogo | Consistencia | `q.normalizar_sku`; lo desconocido va a la clave `-1` |
| Canal escrito de varias formas | Consistencia | la función de canal de la sección B |
| Sucursal vacía en el POS | Completitud | clave `-1` ("Sucursal no informada") |
| Venta sin RUT (cliente no identificado) | Completitud | `rut_anonimizado` nulo → clave `-1` |
| Monto con IVA incluido | Precisión | `monto_neto = monto_total / 1.19`, `monto_iva = monto_total − monto_neto` (ver `docs/acuerdos_tecnicos.md`) |

In [ ]:
# Contrato de salida de asignar_claves(): columnas que espera la carga de fact_ventas (no lo borres)
COLUMNAS_HECHOS = ["id_transaccion", "linea", "fecha_key", "producto_key", "cliente_key", "sucursal_key",
                   "canal_key", "cantidad", "monto_neto", "monto_iva", "monto_total", "devolucion"]

In [ ]:
def transformar_ventas(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """✏️ TODO. Retorna (ventas_validas, rechazos): aún con las columnas de texto, ya limpias y con tipos correctos
    (fecha como datetime, números como numéricos, SKU/canal/sucursal normalizados, `rut_anonimizado`, `monto_neto`,
    `monto_iva`, `devolucion`). Las claves sustitutas se asignan en `asignar_claves`.
    """
    raise NotImplementedError("TODO: implementar transformar_ventas()")

### D.3 Claves sustitutas  ✏️ TODO
Aquí se ve el valor del SCD2: cada venta toma la versión de **producto** y **cliente** vigente **a su fecha**.

In [ ]:
def asignar_claves(ventas: pd.DataFrame) -> pd.DataFrame:
    """✏️ TODO: agregar las claves de las 5 dimensiones y dejar `COLUMNAS_HECHOS`.

    - fecha_key:    fecha en formato AAAAMMDD.
    - producto_key / cliente_key (SCD2): `db.asignar_clave_scd2(ventas, db.leer_tabla(engine, "dim_producto"), "sku", "fecha", "producto_key")`
      (para cliente la clave en la dimensión es `rut_anonimizado`: usa el parámetro `clave_hechos` si tu columna se llama distinto).
    - sucursal_key / canal_key (SCD1): `db.mapa_claves(...)` + `map` + `fillna(-1)`.
    """
    raise NotImplementedError("TODO: implementar asignar_claves()")

### D.4 Load y ejecución
Carga idempotente por `(id_transaccion, linea)`: ejecutar dos veces no duplica.

In [ ]:
def cargar_hechos(hechos: pd.DataFrame, rechazos: pd.DataFrame) -> dict:
    assert set(COLUMNAS_HECHOS) <= set(hechos.columns), "asignar_claves() no devolvió las columnas esperadas"
    db.registrar_rechazos(engine, "fact_ventas", rechazos)
    return db.insertar_si_no_existe(engine, schema.fact_ventas, hechos[COLUMNAS_HECHOS], ["id_transaccion", "linea"])

In [ ]:
cargar_dim_canal()
cargar_dim_sucursal()

with db.Ejecucion(engine, "fact_ventas") as run:
    crudo = extraer_ventas()
    run.leidas = len(crudo)
    validas, rechazos = transformar_ventas(crudo)
    medallion.guardar_capa(validas, "silver", "ventas")
    hechos = asignar_claves(validas)
    resumen = cargar_hechos(hechos, rechazos)
    run.cargadas = resumen["insertadas"]
    run.rechazadas = len(rechazos)
resumen

## 5. Validación
Conciliación de conteos y de montos: lo que entró = lo que se cargó + lo que se rechazó.

In [ ]:
def validar_hechos():
    n_leidas, n_validas, n_rech = len(crudo), len(validas), len(rechazos)
    assert n_leidas == n_validas + n_rech, f"No concilia: {n_leidas} leídas ≠ {n_validas} válidas + {n_rech} rechazadas"
    cargados = db.leer_tabla(engine, "fact_ventas")
    assert cargados.set_index(["id_transaccion", "linea"]).index.is_unique, "hay líneas duplicadas en el DW"
    assert abs(cargados["monto_total"].sum() - hechos["monto_total"].sum()) < 1, "el monto cargado no coincide con el esperado"
    sin_dato = (cargados[["producto_key", "cliente_key", "sucursal_key", "canal_key"]] == -1).any(axis=1).mean()
    print(f"Conciliación OK · {n_leidas} = {n_validas} + {n_rech} · hechos con alguna clave 'No informado': {sin_dato:.1%}")

validar_hechos()

In [ ]:
# Idempotencia: volver a cargar los mismos hechos no inserta nada
filas_antes = len(db.leer_tabla(engine, "fact_ventas"))
r2 = cargar_hechos(hechos, pd.DataFrame(columns=q.COLUMNAS_RECHAZO))
assert len(db.leer_tabla(engine, "fact_ventas")) == filas_antes and r2["insertadas"] == 0
print("Idempotencia OK:", r2)

## 6. Evidencia
Una consulta de negocio sobre el esquema estrella demuestra que el modelo responde las preguntas del caso.

In [ ]:
consulta = """
SELECT g.region, p.categoria, SUM(f.monto_neto) AS venta_neta, SUM(f.cantidad) AS unidades
FROM fact_ventas f
JOIN dim_sucursal s ON s.sucursal_key = f.sucursal_key
JOIN dim_geografia g ON g.geografia_key = s.geografia_key
JOIN dim_producto p ON p.producto_key = f.producto_key
WHERE f.devolucion = 0
GROUP BY g.region, p.categoria
ORDER BY venta_neta DESC
"""
with engine.connect() as conn:
    display(pd.read_sql(consulta, conn).head(10))

In [ ]:
# Trazabilidad: ejecuciones y rechazos de TODO el pipeline
ejec = db.leer_tabla(engine, "auditoria_ejecuciones")
display(ejec.tail(10)[["proceso", "inicio", "estado", "filas_leidas", "filas_cargadas", "filas_rechazadas"]])
rech = db.leer_tabla(engine, "auditoria_rechazos")
rech.groupby(["proceso", "motivo_rechazo"]).size().rename("filas").to_frame()

## 7. Decisiones de diseño (completar)
- Ventas anuladas: ¿rechazo o carga marcada? ¿Por qué?
- Devoluciones como columna (`devolucion`) vs. tabla de hechos separada (`Fact_Devoluciones` del EP1): ¿qué ganamos y qué perdemos?
- Claves `-1`: ¿qué porcentaje de hechos queda con alguna dimensión «No informada» y qué significa para el análisis?